# 08 · Exits and entries: momentum or reversal?

Idea from Conrad & Yavuz (*Momentum and Reversal*), applied to S&P 500 index changes:

- stocks that **exit** the S&P 500 should be past **losers**; stocks that **enter** should be past **winners**;
- in the six months after the event, some keep going (momentum) and some turn around (reversal).

The notebook answers three questions:

1. **Q1.** Are exits losers and entries winners? Formation return over months t−12 to t−2 (month t−1 skipped), ranked against all eligible CRSP stocks and against S&P 500 members at t−1.
2. **Q2.** What share reverses and what share continues over months t+1 to t+6? Returns are adjusted with the CRSP value-weighted market (`vwretd`). An exit *reverses* if it beats the market; an entry *reverses* if it trails it.
3. **Q3.** Calendar-time, equal-weighted portfolios held over months t+1 to t+6, with CAPM and FF5 + momentum alphas (Newey–West, 6 lags).

Month t is the effective month. As in notebook 02, the headline sample ends in December 2022 and 2023–2025 is provisional. Additions flagged by `likely_spin_or_merger` are excluded. Exits after which the stock stops trading (mostly takeovers) are excluded, because they leave the index for reasons unrelated to past returns.

**Warning.** The portfolios marked **EX POST** choose stocks with returns from the holding window itself. They are an upper bound on what a perfect classifier could earn, not an investable strategy.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"

with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

HOLD = 6                       # holding window: months t+1 ... t+6
DELIST_PENALTY = -0.30         # Shumway (1997): return assumed when a delisting return is missing
LOSER_CUT, WINNER_CUT = 0.30, 0.70
NW_LAGS = HOLD
PERIOD_BINS = [1974, 1999, 2009, 2017, 2022, 2025]
PERIOD_LABELS = ["1975–1999", "2000–2009", "2010–2017", "2018–2022", "2023–2025 (provisional)"]

## Monthly panel: formation returns and market

`mom_12_2` compounds months t−12 to t−2 and requires eleven consecutive monthly returns, the same convention as `trailing` in notebook 04.

Delisting returns are handled two ways:

- `ret`: a return with a partial or missing delisting return (flags `G`, `M`, `V`) is set to missing, as in notebook 02. If those delistings are mostly for poor performance, this **overstates** the returns of exits.
- `ret_cons` (conservative): the delisting month gets the observed return compounded with a −30% delisting return, the average for performance delistings in Shumway (1997). Results under both are reported for exits.

In [ ]:
p = monthly_panel[["permno", "mthcaldt", "mthret", "mthdelflg", "in_sp500", "eligible"]].copy()
p = p.sort_values(["permno", "mthcaldt"], ignore_index=True)
p["month"] = p["mthcaldt"].dt.to_period("M")
p["mnum"] = p["mthcaldt"].dt.year * 12 + p["mthcaldt"].dt.month
partial_delist = p["mthdelflg"].isin(["G", "M", "V"])
p["ret"] = p["mthret"].mask(partial_delist)
p["ret_cons"] = p["mthret"].where(~partial_delist, (1 + p["mthret"].fillna(0)) * (1 + DELIST_PENALTY) - 1)
by_stock = p.groupby("permno")


def months_since(lag):
    return p["mnum"] - by_stock["mnum"].shift(lag)


log_ret = np.log1p(p["ret"])
sum_11 = log_ret.groupby(p["permno"]).rolling(11, min_periods=11).sum().droplevel(0)
sum_11 = sum_11.where(months_since(10).eq(10))                 # 11 consecutive months
p["mom_12_2"] = np.expm1(sum_11.groupby(p["permno"]).shift(2).where(months_since(2).eq(2)))
p["member_prev"] = by_stock["in_sp500"].shift(1).where(months_since(1).eq(1)).fillna(False).astype(bool)

# percentile of mom_12_2 among eligible stocks in the same month
elig = p["eligible"] & p["mom_12_2"].notna()
p.loc[elig, "pct_all"] = p.loc[elig].groupby("month")["mom_12_2"].rank(pct=True)

market = pd.read_csv(paths["index_monthly"], usecols=["mthcaldt", "vwretd"], parse_dates=["mthcaldt"])
market = market.assign(month=market["mthcaldt"].dt.to_period("M")).set_index("month")["vwretd"]
returns = p.set_index(["permno", "month"])[["ret", "ret_cons"]]
print(f"Panel: {len(p):,} rows, {p['permno'].nunique():,} PERMNOs, "
      f"{p['month'].min()} to {p['month'].max()}")

## Event register: entries and exits

**Entries** come from `events` (notebook 01) without contaminated additions. **Exits** through December 2022 are the end dates of the membership spells. An end date is not an exit when the same PERMNO starts a new spell within five days. Exits in 2023–2025 come from the supplied removals file, matched to a PERMNO with the ±10-day rule of notebook 01 and kept only when the match is unique.

In [ ]:
entries = events.loc[events["permno"].notna() & ~events["likely_spin_or_merger"],
                     ["permno", "effective_date"]].assign(type="entry", source="events")

spells = panel_spells.dropna(subset=["permno"]).astype({"permno": "int64"}).sort_values(["permno", "start"])
next_start = spells.groupby("permno")["start"].shift(-1)
real_end = spells["end"].lt(history_end) & ~(next_start - spells["end"]).dt.days.le(5)
historical_exits = spells.loc[real_end, ["permno", "end"]].rename(columns={"end": "effective_date"}) \
    .assign(type="exit", source="membership_spells")

removal_matches = ticker_candidates(recent_removals, tolerance_days=10)
unique_ids = removal_matches.groupby("event_id").size().loc[lambda s: s.eq(1)].index
recent_exits = recent_removals.loc[unique_ids, ["effective_date"]].assign(
    permno=removal_matches.set_index("event_id").loc[unique_ids, "permno"],
    type="exit", source="removals_2023_2025",
)

ev = pd.concat([entries, historical_exits, recent_exits], ignore_index=True)
ev["permno"] = ev["permno"].astype("int64")
ev["month"] = ev["effective_date"].dt.to_period("M")
ev = ev.drop_duplicates(["permno", "month", "type"])
ev = ev.loc[ev["month"].dt.year.ge(1976) & (ev["month"] + HOLD).le(p["month"].max())].reset_index(drop=True)
ev["period"] = pd.cut(ev["month"].dt.year, bins=PERIOD_BINS, labels=PERIOD_LABELS)

## Membership after December 2022 (provisional)

`in_sp500` from notebook 01 stops at the verified boundary, December 2022. To rank 2023–2025 events against S&P 500 peers, membership is rolled forward month by month: members at the boundary, plus recent additions, minus recent exits, in order of effective date. Without this step, the 2023–2025 percentiles would all be missing.

In [ ]:
# last month with a full verified index (about 500 members); should be December 2022
member_counts = p.loc[p["in_sp500"]].groupby("month").size()
boundary = member_counts.loc[member_counts.ge(0.8 * member_counts.max())].index.max()
members = set(p.loc[p["month"].eq(boundary) & p["in_sp500"], "permno"])
recent_changes = pd.concat([
    events.loc[events["permno"].notna() & events["effective_date"].gt(history_end), ["permno", "effective_date"]]
          .assign(change=+1),
    recent_exits[["permno", "effective_date"]].assign(change=-1),
]).astype({"permno": "int64"}).assign(month=lambda d: d["effective_date"].dt.to_period("M")).sort_values("effective_date")
members_at = {boundary: frozenset(members)}
for m in pd.period_range(boundary + 1, p["month"].max(), freq="M"):
    for row in recent_changes.loc[recent_changes["month"].eq(m)].itertuples():
        (members.add if row.change > 0 else members.discard)(row.permno)
    members_at[m] = frozenset(members)

after = p["month"].gt(history_end.to_period("M"))   # never overwrite verified membership
p.loc[after, "member_prev"] = [perm in members_at[m - 1] for perm, m in zip(p.loc[after, "permno"], p.loc[after, "month"])]
print(f"Starting membership: {len(members_at[boundary])} stocks at {boundary} (should be about 500 at 2022-12).")
print("Provisional S&P 500 membership size, end of each year:",
      {str(m): len(s) for m, s in members_at.items() if m.month == 12 and m > history_end.to_period("M")})

# S&P 500 peers: stocks that were members at t-1, with a formation return at t
peers = (p.loc[p["member_prev"] & p["mom_12_2"].notna()]
         .groupby("month")["mom_12_2"].apply(lambda s: np.sort(s.to_numpy())))

## Event-level returns

For each event: formation return and percentiles at month t, and the six-month holding return compounded over months t+1 to t+6. A stock that stops trading earlier keeps the months it has (`hold_months` < 6).

In [ ]:
at_t = p.set_index(["permno", "month"])[["mom_12_2", "pct_all"]]
ev = ev.join(at_t, on=["permno", "month"])


def pct_among_peers(row):
    arr = peers.get(row["month"])
    if arr is None or len(arr) == 0 or pd.isna(row["mom_12_2"]):
        return np.nan
    return np.searchsorted(arr, row["mom_12_2"]) / len(arr)


ev["pct_sp500"] = ev.apply(pct_among_peers, axis=1)

hold = ev[["permno", "month"]].reset_index(names="event_id").merge(
    pd.DataFrame({"k": range(1, HOLD + 1)}), how="cross")
hold["month"] = hold["month"] + hold["k"]
hold[["ret", "ret_cons"]] = returns.reindex(pd.MultiIndex.from_frame(hold[["permno", "month"]])).to_numpy()
hold["mkt"] = market.reindex(hold["month"]).to_numpy()


def buy_and_hold(col, suffix):
    traded = hold.dropna(subset=[col])
    out = traded.groupby("event_id").agg(**{
        f"hold_ret{suffix}": (col, lambda r: np.prod(1 + r) - 1),
        f"hold_months{suffix}": (col, "size"),
    })
    out[f"hold_mkt{suffix}"] = traded.groupby("event_id")["mkt"].apply(lambda r: np.prod(1 + r) - 1)
    out[f"hold_ar{suffix}"] = out[f"hold_ret{suffix}"] - out[f"hold_mkt{suffix}"]
    return out


ev = ev.join(buy_and_hold("ret", "")).join(buy_and_hold("ret_cons", "_cons"))
ev["traded_after"] = hold.loc[hold["k"].eq(1)].set_index("event_id")["ret_cons"].notna()

sample = ev.loc[ev["traded_after"] & ev["mom_12_2"].notna()].copy()
display(pd.DataFrame({
    "Events": ev.groupby("type").size(),
    "Still trading at t+1": ev.loc[ev["traded_after"]].groupby("type").size(),
    "With formation return (final sample)": sample.groupby("type").size(),
}))
display(sample.groupby(["period", "type"], observed=True).size().unstack())

## Q1 · Are exits losers and entries winners?

Loser: bottom 30% of formation returns. Winner: top 30%. The comparison with S&P 500 members is the relevant one: an exit is a loser *relative to the index it leaves*.

In [ ]:
def share(x, condition):
    """Percentage of events meeting the condition, among events where x is observed."""
    return 100 * condition[x.notna()].mean() if x.notna().any() else np.nan


def q1_stats(g):
    return pd.Series({
        "N": len(g),
        "mean formation return": g["mom_12_2"].mean(),
        "median formation return": g["mom_12_2"].median(),
        "mean percentile, all CRSP": g["pct_all"].mean(),
        "% losers, all CRSP": share(g["pct_all"], g["pct_all"].le(LOSER_CUT)),
        "% winners, all CRSP": share(g["pct_all"], g["pct_all"].ge(WINNER_CUT)),
        "N with S&P 500 percentile": g["pct_sp500"].notna().sum(),
        "mean percentile, S&P 500": g["pct_sp500"].mean(),
        "% losers, S&P 500": share(g["pct_sp500"], g["pct_sp500"].le(LOSER_CUT)),
        "% winners, S&P 500": share(g["pct_sp500"], g["pct_sp500"].ge(WINNER_CUT)),
    })


headline = sample["month"].dt.year.le(2022)
q1 = sample.loc[headline].groupby("type").apply(q1_stats, include_groups=False).T
q1_by_period = sample.groupby(["type", "period"], observed=True).apply(q1_stats, include_groups=False)
print("Headline sample 1976–2022")
display(q1.round(3))
display(q1_by_period[["N", "N with S&P 500 percentile", "mean percentile, S&P 500",
                      "% losers, S&P 500", "% winners, S&P 500"]].round(2))

## Q2 · Reversal or continuation over months t+1 to t+6?

Market-adjusted return = stock buy-and-hold return − market buy-and-hold return over the same months. Expect shares close to 50% even if the mean effect is real: six-month idiosyncratic risk is large. Compare the **mean** and the **median** too.

In [ ]:
sample["reversal"] = np.where(sample["type"].eq("exit"), sample["hold_ar"].gt(0), sample["hold_ar"].lt(0))
sample["is_loser"] = sample["pct_sp500"].le(LOSER_CUT)
sample["is_winner"] = sample["pct_sp500"].ge(WINNER_CUT)


def q2_stats(g):
    return pd.Series({
        "N": len(g),
        "% reversal": 100 * g["reversal"].mean(),
        "% continuation": 100 * (1 - g["reversal"].mean()),
        "mean 6m return": g["hold_ret"].mean(),
        "mean 6m market-adjusted": g["hold_ar"].mean(),
        "median 6m market-adjusted": g["hold_ar"].median(),
        "t-stat of mean market-adjusted": g["hold_ar"].mean() / g["hold_ar"].std() * np.sqrt(len(g)),
        "% with fewer than 6 months": 100 * g["hold_months"].lt(HOLD).mean(),
        "mean 6m mkt-adj, −30% delisting": g["hold_ar_cons"].mean(),
        "median 6m mkt-adj, −30% delisting": g["hold_ar_cons"].median(),
        "% reversal, −30% delisting": 100 * np.where(g["type"].eq("exit"), g["hold_ar_cons"].gt(0),
                                                       g["hold_ar_cons"].lt(0)).mean(),
    })


is_exit, is_entry = sample["type"].eq("exit"), sample["type"].eq("entry")
groups = {
    "All exits": is_exit,
    "Exits that are losers": is_exit & sample["is_loser"],
    "All entries": is_entry,
    "Entries that are winners": is_entry & sample["is_winner"],
}
q2 = pd.DataFrame({name: q2_stats(sample.loc[mask & headline]) for name, mask in groups.items()})
q2_by_period = sample.groupby(["type", "period"], observed=True).apply(lambda g: q2_stats(g.assign(type=g.name[0])), include_groups=False)
print("Headline sample 1976–2022. Event-level t-stats ignore overlap between events in the same month.")
display(q2.round(3))
display(q2_by_period[["N", "% reversal", "mean 6m market-adjusted", "median 6m market-adjusted",
                      "mean 6m mkt-adj, −30% delisting"]].round(3))

## Q3 · Calendar-time portfolios

Each month, an equal-weighted basket of all stocks in months t+1 to t+6 after their event. Investable baskets use only information available at the event. **EX POST** baskets select stocks with the holding-window outcome and are not investable.

In [ ]:
def basket(event_ids, col="ret"):
    h = hold.loc[hold["event_id"].isin(event_ids)].dropna(subset=[col])
    return h.groupby("month").agg(ret=(col, "mean"), n=(col, "size"))


ids = lambda mask: sample.index[mask]
baskets = {
    "Long exits": basket(ids(is_exit)),
    "Long loser exits": basket(ids(is_exit & sample["is_loser"])),
    "Long exits (−30% delisting)": basket(ids(is_exit), "ret_cons"),
    "Long loser exits (−30% delisting)": basket(ids(is_exit & sample["is_loser"]), "ret_cons"),
    "Long entries": basket(ids(is_entry)),
    "EX POST: exits that reversed": basket(ids(is_exit & sample["reversal"])),
    "EX POST: entries that kept winning": basket(ids(is_entry & ~sample["reversal"])),
    "EX POST: reversal + momentum": basket(ids((is_exit & sample["reversal"]) | (is_entry & ~sample["reversal"]))),
}
calendar = pd.period_range(p["month"].min(), p["month"].max(), freq="M", name="month")
basket_returns = pd.DataFrame({k: v["ret"] for k, v in baskets.items()}).reindex(calendar)
basket_counts = pd.DataFrame({k: v["n"] for k, v in baskets.items()}).reindex(calendar)
basket_returns["Long exits − short entries"] = basket_returns["Long exits"] - basket_returns["Long entries"]
display(basket_counts.describe().loc[["count", "mean", "min", "max"]].round(1))

In [ ]:
def newey_west(y, X, lags=NW_LAGS):
    X = np.column_stack([np.ones(len(y)), X])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    scores = X * (y - X @ beta)[:, None]
    S = scores.T @ scores
    for lag in range(1, lags + 1):
        g = scores[lag:].T @ scores[:-lag]
        S += (1 - lag / (lags + 1)) * (g + g.T)
    bread = np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(bread @ S @ bread))
    return beta, beta / se


factors = None
ff5_path, mom_path = data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip", data_dir / "F-F_Momentum_Factor_CSV.zip"
if ff5_path.is_file() and mom_path.is_file():
    factors = read_monthly_factors(ff5_path, ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]).join(
        read_monthly_factors(mom_path, ["Mom"]), how="inner")
else:
    print("Factor archives missing in Data/: showing raw returns only.")

FACTORS = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]


def summarize(r, name, months_mask):
    r = r.loc[months_mask].dropna()
    out = {"months": len(r), "avg # stocks": basket_counts[name].loc[r.index].mean() if name in basket_counts else np.nan,
           "mean monthly %": 100 * r.mean(), "ann. vol %": 100 * r.std() * np.sqrt(12)}
    if len(r) < 24:                      # too few months for reliable regressions
        return out
    out["t (NW)"] = newey_west(r.to_numpy(), np.empty((len(r), 0)))[1][0]
    if factors is not None:
        d = pd.concat([r.rename("r"), factors], axis=1, join="inner")
        y = d["r"] if "short" in name else d["r"] - d["RF"]
        out["Sharpe (ann.)"] = y.mean() / y.std() * np.sqrt(12)
        b, t = newey_west(y.to_numpy(), d[["Mkt-RF"]].to_numpy())
        out["CAPM alpha %"], out["CAPM t"] = 100 * b[0], t[0]
        b, t = newey_west(y.to_numpy(), d[FACTORS].to_numpy())
        out["FF5+Mom alpha %"], out["FF5+Mom t"] = 100 * b[0], t[0]
        out["beta Mom"] = b[-1]
    return out


for label, mask in [("Headline 1976–2022", calendar.year <= 2022),
                    ("Robustness 1976–2025 (provisional after 2022)", calendar.year <= 2025)]:
    print(label)
    display(pd.DataFrame({c: summarize(basket_returns[c], c, mask) for c in basket_returns}).T.round(3))

In [ ]:
by_period = {}
period_of_month = pd.cut(calendar.year, bins=PERIOD_BINS, labels=PERIOD_LABELS)
for c in ["Long exits", "Long exits (−30% delisting)", "Long loser exits", "Long entries",
          "Long exits − short entries"]:
    for per in PERIOD_LABELS:
        by_period[(c, per)] = summarize(basket_returns[c], c, period_of_month == per)
display(pd.DataFrame(by_period).T.round(3))